# ⛽📈 **Парсинг внешних данных(экономические показатели, данные о запасах в хранилищах, данные о ценах на нефть и газ.):** 

![Нефть и газ](https://avatars.mds.yandex.net/i?id=f0c6e485cc98b2d1ee4633e48d076c80_l-5315578-images-thumbs&n=13.jpg)

# <center> 🔄 Загрузка библиотек <center>

In [3]:
import numpy as np 
import pandas as pd

import requests

from fredapi import Fred

In [5]:
def get_eia_data(url, api_key, frequency, facets_name, facets_value):

    """Получаем цены природный газ"""

    params = {
        'api_key': api_key,
        'frequency': frequency,
        'data[0]': 'value',
        'start': '2015-01',
        'end': '2025-09',
        'sort[0][column]': 'period',
        'sort[0][direction]': 'desc',
        'offset': 0,
        'length': 5000
    }

    for i, (name, value) in enumerate(zip(facets_name, facets_value)):
        params[f'facets[{name}][]'] = value

    response = requests.get(url, params=params)

    if response.status_code == 200:
        data = response.json()
        df = pd.DataFrame(data['response']['data'])
        df = df.astype({'value': float})
        return df
    else: 
        print(f"Error: {response.status_code}")
        print(f"URL: {response.url}")
        return None

gas_prices_df = get_eia_data(
    url='https://api.eia.gov/v2/natural-gas/pri/fut/data/', 
    api_key=EIA_API_KEY, 
    frequency='monthly', 
    facets_name=['process'], 
    facets_value=['PS0']
)

brent_prices_df = get_eia_data(
    url='https://api.eia.gov/v2/petroleum/pri/spt/data/', 
    api_key=EIA_API_KEY, 
    frequency='monthly', 
    facets_name=['product'], 
    facets_value=['EPCBRENT']
)

wti_prices_df = get_eia_data(
    url='https://api.eia.gov/v2/petroleum/pri/spt/data/', 
    api_key=EIA_API_KEY, 
    frequency='monthly', 
    facets_name=['product'], 
    facets_value=['EPCWTI']
)

co2_emissions_df = get_eia_data(
    url='https://api.eia.gov/v2/seds/data/', 
    api_key=EIA_API_KEY, 
    frequency='annual', 
    facets_name=['stateId', 'seriesId'], 
    facets_value=['US', ['CDTCR', 'CDTPR', 'FFTCE', 'NNTCE', 'PMTCE']]
)

oil_storage_df = get_eia_data(
    url='https://api.eia.gov/v2/petroleum/stoc/wstk/data/', 
    api_key=EIA_API_KEY, 
    frequency='weekly', 
    facets_name=['product', 'series'], 
    facets_value=['EPC0', ['WCESTUS1', 'W_EPC0_SAX_YCUOK_MBBL']]
)

gas_storage_df = get_eia_data(
    url='https://api.eia.gov/v2/natural-gas/stor/wkly/data/', 
    api_key=EIA_API_KEY, 
    frequency='weekly', 
    facets_name=['series'], 
    facets_value=['NW2_EPG0_SWO_R48_BCF']
)

In [6]:
def convert_weekly_to_monthly(storage_df):
    """Преобразуем недельные данные в месячные"""

    storage_df['period'] = pd.to_datetime(storage_df['period'])

    storage_df['year_month'] = storage_df['period'].dt.to_period('M')

    monthly_storage = storage_df.sort_values('period').groupby(
        ['year_month', 'series-description']
    ).agg({
        'value': 'last'
    }).reset_index()

    monthly_storage.rename(columns={'year_month': 'period'}, inplace=True)

    print(f"После агрегации: {len(monthly_storage)} месячных записей")

    return monthly_storage

# Преобразуем данные
oil_storage_monthly = convert_weekly_to_monthly(oil_storage_df)
gas_storage_monthly = convert_weekly_to_monthly(gas_storage_df)

После агрегации: 256 месячных записей
После агрегации: 128 месячных записей


In [7]:
def get_economic_indicators():
    """Получаем макроэкономические показатели из FRED"""
    fred_series = {
        'GDP': 'GDP',                          # ВВП
        'INDPRO': 'INDPRO',                    # Индекс промпроизводства
        'UNRATE': 'UNRATE',                    # Уровень безработицы
        'CPIAUCSL': 'CPIAUCSL',                # Инфляция (CPI)
        'DTWEXBGS': 'DTWEXBGS',                # Курс доллара
        'UMCSENT': 'UMCSENT',                  # Индекс потребительского доверия
        #'IPG211111CS': 'IPG211111CS',          # Индекс добычи сырой нефти
    }
    
    economic_data = []
    
    for name, series_id in fred_series.items():
        url = f"https://api.stlouisfed.org/fred/series/observations"
        params = {
            'series_id': series_id,
            'api_key': FRED_API_KEY,
            'file_type': 'json',
            'observation_start': '2015-01-01',
            'observation_end': '2025-09-30',
            'frequency': 'm'  # месячные данные
        }
        
        response = requests.get(url, params=params)
        if response.status_code == 200:
            data = response.json()
            df = pd.DataFrame(data['observations'])
            df['value'] = pd.to_numeric(df['value'], errors='coerce')
            df = df[['date', 'value']].rename(columns={'value': name})
            economic_data.append(df)
            print(f"Получены данные {name}: {len(df)} записей")
    
    # Объединяем все экономические показатели
    if economic_data:
        economic_df = economic_data[0]
        for df in economic_data[1:]:
            economic_df = economic_df.merge(df, on='date', how='outer')
        return economic_df
    return None

# Получаем экономические данные
economic_indicators = get_economic_indicators()

Получены данные INDPRO: 129 записей
Получены данные UNRATE: 129 записей
Получены данные CPIAUCSL: 129 записей
Получены данные DTWEXBGS: 129 записей
Получены данные UMCSENT: 129 записей


In [8]:
# Сохраняем
brent_prices_df.to_csv('../data/external/brent_prices.csv', index=False)
wti_prices_df.to_csv('../data/external/wti_prices.csv', index=False)
gas_prices_df.to_csv('../data/external/gas_prices.csv', index=False)
oil_storage_monthly.to_csv('../data/external/oil_storage.csv', index=False)
gas_storage_monthly.to_csv('../data/external/gas_storage.csv', index=False)
co2_emissions_df.to_csv('../data/external/co2_emissions_df.csv', index=False)
economic_indicators.to_csv('../data/external/economic_indicators.csv', index=False)